# Reviewer visualizations for chess cheat-detection outputs

## tl;dr

This notebook turns the project JSON outputs into reviewer-friendly charts and a small Markdown report.

It supports two levels of detail:

1. **Current summary JSONs**: benchmark/profile comparison charts, including ACPL by Elo and Stockfish-vs-Maia scatter.
2. **Richer JSONs from future reruns**: per-game timelines, phase breakdowns, and move-level review tables.

If a chart needs data that is not yet present in the JSON files, the notebook explains what is missing instead of pretending the chart exists.


## Context & Methods

The charts are designed for human review, not automatic conviction.

- **ACPL by Elo range** checks whether a profiled sample is unusually clean compared with its Elo benchmark.
- **Stockfish match % vs Maia match %** looks for the stronger combined signal: objectively strong moves that are less human-like for the claimed Elo range.
- **Per-game timelines** show whether performance is stable or changes suddenly across games.
- **Phase breakdowns** help detect selective help, especially in sharp middlegames.
- **Move-level tables** give a reviewer the actual moves to inspect.

The notebook writes chart files into `reports/visualization_report/figures/` and a Markdown report into `reports/visualization_report/README.md`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import html
import json
import math
import re
import statistics
from typing import Any

from PIL import Image, ImageDraw, ImageFont

try:
    from IPython.display import HTML, Image as NotebookImage, display
except Exception:  # pragma: no cover - only used outside notebooks
    HTML = NotebookImage = None
    def display(value):
        print(value)

# Parameters you can change before rerunning the notebook.
PROJECT_ROOT = Path.cwd()
DATASET_KIND = "standard"  # "standard" or "broadcast"
PROFILE_RESULT_PATH = None  # Example: "results/standard_games/result_[1400, 1599].json"
REPORT_DIR = PROJECT_ROOT / "reports" / "visualization_report"
FIGURE_DIR = REPORT_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset kind: {DATASET_KIND}")
print(f"Report folder: {REPORT_DIR}")


In [ ]:
from __future__ import annotations

def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def normalize_key(key: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", key.strip().lower()).strip("_")


def get_any(data: dict[str, Any], *keys: str, default: Any = None) -> Any:
    normalized = {normalize_key(k): v for k, v in data.items()}
    for key in keys:
        normalized_key = normalize_key(key)
        if normalized_key in normalized:
            return normalized[normalized_key]
    return default


def parse_range_from_filename(path: Path) -> list[int] | None:
    numbers = [int(n) for n in re.findall(r"\d+", path.stem)]
    if len(numbers) >= 2:
        return numbers[-2:]
    return None


def elo_label(elo_range: list[int]) -> str:
    return f"{elo_range[0]}–{elo_range[1]}"


def load_benchmarks(kind: str) -> list[dict[str, Any]]:
    rows = []
    for path in sorted((PROJECT_ROOT / "benchmark_data" / kind).glob("benchmark_*.json")):
        raw = load_json(path)
        elo_range = get_any(raw, "elo_range", default=parse_range_from_filename(path))
        if not elo_range:
            continue
        rows.append({
            "path": path,
            "kind": kind,
            "elo_range": elo_range,
            "elo_low": elo_range[0],
            "elo_high": elo_range[1],
            "elo_label": elo_label(elo_range),
            "games_analyzed": get_any(raw, "games_analyzed", default=0),
            "player_side_samples": get_any(raw, "player_side_samples", default=0),
            "avg_acpl": get_any(raw, "avg_acpl", default=0.0),
            "std_acpl": get_any(raw, "std_acpl", default=math.sqrt(get_any(raw, "variance_acpl", default=0.0))),
            "avg_accuracy": get_any(raw, "avg_accuracy", default=0.0),
            "std_accuracy": get_any(raw, "std_accuracy", default=math.sqrt(get_any(raw, "variance_accuracy", default=0.0))),
            "avg_match_pct": get_any(raw, "avg_match_pct", default=0.0),
            "std_match_pct": get_any(raw, "std_match_pct", default=math.sqrt(get_any(raw, "variance_match_pct", default=0.0))),
            "avg_maia_pct": get_any(raw, "avg_maia_pct", default=0.0),
            "std_maia_pct": get_any(raw, "std_maia_pct", default=math.sqrt(get_any(raw, "variance_maia_pct", default=0.0))),
            "player_side_summaries": get_any(raw, "player_side_summaries", default=[]),
        })
    return sorted(rows, key=lambda row: (row["elo_low"], row["elo_high"]))


def load_profiles(kind: str) -> list[dict[str, Any]]:
    folder = "standard_games" if kind == "standard" else "broadcast_games"
    rows = []
    paths = []
    if PROFILE_RESULT_PATH:
        paths = [PROJECT_ROOT / PROFILE_RESULT_PATH]
    else:
        paths = sorted((PROJECT_ROOT / "results" / folder).glob("result_*.json"))

    for path in paths:
        if not path.exists():
            continue
        raw = load_json(path)
        elo_range = get_any(raw, "Profile", "elo_range", default=parse_range_from_filename(path))
        if not elo_range:
            continue
        suspicion = get_any(raw, "SUSPICION SCORE", default={}) or {}
        rows.append({
            "path": path,
            "kind": kind,
            "elo_range": elo_range,
            "elo_low": elo_range[0],
            "elo_high": elo_range[1],
            "elo_label": elo_label(elo_range),
            "games_analyzed": get_any(raw, "Games_analysed", "Games analysed", "games", default=0),
            "avg_acpl": get_any(raw, "avg ACPL", "avg_acpl", default=0.0),
            "acpl_stdev": get_any(raw, "std_deviaiton", "std_deviation", "acpl_stdev", default=0.0),
            "avg_match_pct": get_any(raw, "avg top-move %", "avg_match_pct", default=0.0),
            "avg_maia_pct": get_any(raw, "avg_maia_move_matching %", "avg_maia_pct", default=0.0),
            "avg_accuracy": get_any(raw, "avg accuracy", "avg_accuracy", default=0.0),
            "confidence_pct": get_any(raw, "confidence %", "confidence", default=0.0),
            "suspicion": suspicion,
            "per_game": get_any(raw, "per_game", default=[]),
            "move_review": get_any(raw, "move_review", default=[]),
        })
    return sorted(rows, key=lambda row: (row["elo_low"], row["elo_high"]))

benchmarks = load_benchmarks(DATASET_KIND)
profiles = load_profiles(DATASET_KIND)

print(f"Loaded {len(benchmarks)} benchmark files.")
print(f"Loaded {len(profiles)} profile result files.")
if benchmarks:
    print(f"Benchmark ranges: {benchmarks[0]['elo_label']} to {benchmarks[-1]['elo_label']}")
if profiles:
    print(f"Profile ranges: {profiles[0]['elo_label']} to {profiles[-1]['elo_label']}")


In [ ]:
from __future__ import annotations

# Lightweight PNG chart helpers. These avoid pandas/matplotlib so the report is easy to rerun.

BLUE = (31, 119, 180)
LIGHT_BLUE = (158, 202, 225)
ORANGE = (217, 95, 2)
ORANGE_DARK = (127, 39, 4)
PURPLE = (117, 107, 177)
GREEN = (44, 160, 44)
INK = (34, 34, 34)
MID = (85, 85, 85)
GRID = (232, 232, 232)
WHITE = (255, 255, 255)
REVIEW_FILL = (253, 208, 162)


def font(size: int, bold: bool = False) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/Library/Fonts/Arial Bold.ttf" if bold else "/Library/Fonts/Arial.ttf",
        "/System/Library/Fonts/Helvetica.ttc",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size=size)
        except Exception:
            continue
    return ImageFont.load_default()


FONT_10 = font(10)
FONT_11 = font(11)
FONT_12 = font(12)
FONT_13 = font(13)
FONT_16 = font(16, bold=True)
FONT_22 = font(22, bold=True)


def show_png(path: Path):
    if NotebookImage is not None:
        display(NotebookImage(filename=str(path)))
    else:
        print(path)


def show_html(html_text: str):
    if HTML is not None:
        display(HTML(html_text))
    else:
        print(html_text[:500] + "...")


def write_text(path: Path, content: str) -> Path:
    path.write_text(content, encoding="utf-8")
    return path


def nice_domain(values: list[float], floor: float | None = None, ceiling: float | None = None) -> tuple[float, float]:
    clean = [v for v in values if isinstance(v, (int, float)) and math.isfinite(v)]
    if not clean:
        return (0.0, 1.0)
    lo, hi = min(clean), max(clean)
    if lo == hi:
        lo -= 1
        hi += 1
    padding = (hi - lo) * 0.08
    lo -= padding
    hi += padding
    if floor is not None:
        lo = min(lo, floor)
    if ceiling is not None:
        hi = max(hi, ceiling)
    return lo, hi


def scale(value: float, src: tuple[float, float], dst: tuple[float, float]) -> float:
    lo, hi = src
    a, b = dst
    if hi == lo:
        return (a + b) / 2
    return a + (value - lo) * (b - a) / (hi - lo)


def axis_labels(domain: tuple[float, float], count: int = 5) -> list[float]:
    lo, hi = domain
    if count <= 1:
        return [lo]
    return [lo + i * (hi - lo) / (count - 1) for i in range(count)]


def text_size(draw: ImageDraw.ImageDraw, text: str, text_font: ImageFont.ImageFont) -> tuple[int, int]:
    box = draw.textbbox((0, 0), text, font=text_font)
    return box[2] - box[0], box[3] - box[1]


def draw_centered_text(draw: ImageDraw.ImageDraw, xy: tuple[float, float], text: str, text_font: ImageFont.ImageFont, fill=INK):
    w, h = text_size(draw, text, text_font)
    draw.text((xy[0] - w / 2, xy[1] - h / 2), text, font=text_font, fill=fill)


def draw_rotated_text(image: Image.Image, xy: tuple[int, int], text: str, text_font: ImageFont.ImageFont, fill=INK):
    temp = Image.new("RGBA", (240, 34), (255, 255, 255, 0))
    temp_draw = ImageDraw.Draw(temp)
    temp_draw.text((0, 0), text, font=text_font, fill=fill + (255,))
    rotated = temp.rotate(90, expand=True)
    image.alpha_composite(rotated, (xy[0], xy[1]))


def save_png(image: Image.Image, path: Path) -> Path:
    image.convert("RGB").save(path, "PNG", optimize=True)
    show_png(path)
    return path


In [ ]:
from __future__ import annotations

def acpl_distribution_chart(benchmarks: list[dict[str, Any]], profiles: list[dict[str, Any]]) -> Path | None:
    if not benchmarks:
        print("No benchmark rows available for ACPL chart.")
        return None

    width, height = 1240, 760
    left, right, top, bottom = 110, 250, 95, 170
    plot_w, plot_h = width - left - right, height - top - bottom
    image = Image.new("RGBA", (width, height), WHITE + (255,))
    draw = ImageDraw.Draw(image)

    values = []
    for row in benchmarks:
        values.extend([row["avg_acpl"] - row["std_acpl"], row["avg_acpl"] + row["std_acpl"]])
        for summary in row.get("player_side_summaries", []):
            values.append(summary.get("acpl", row["avg_acpl"]))
    values.extend([row["avg_acpl"] for row in profiles])
    y_domain = nice_domain(values, floor=0)
    x_step = plot_w / max(1, len(benchmarks) - 1)
    profile_by_range = {(p["elo_low"], p["elo_high"]): p for p in profiles}

    draw.text((left, 28), "ACPL by Elo range", font=FONT_22, fill=INK)
    draw.text((left, 58), "Benchmark mean ± standard deviation; profile marker overlaid when present.", font=FONT_13, fill=MID)

    for tick in axis_labels(y_domain):
        y = scale(tick, y_domain, (top + plot_h, top))
        draw.line((left, y, left + plot_w, y), fill=GRID, width=1)
        label = f"{tick:.0f}"
        w, h = text_size(draw, label, FONT_11)
        draw.text((left - w - 10, y - h / 2), label, font=FONT_11, fill=MID)

    for index, row in enumerate(benchmarks):
        x = left + index * x_step
        mean = row["avg_acpl"]
        std = row["std_acpl"]
        y_mean = scale(mean, y_domain, (top + plot_h, top))
        y_low = scale(mean - std, y_domain, (top + plot_h, top))
        y_high = scale(mean + std, y_domain, (top + plot_h, top))

        summaries = row.get("player_side_summaries", [])
        for dot_index, summary in enumerate(summaries[:300]):
            jitter = ((dot_index % 11) - 5) * 2.0
            y_dot = scale(summary.get("acpl", mean), y_domain, (top + plot_h, top))
            draw.ellipse((x + jitter - 2, y_dot - 2, x + jitter + 2, y_dot + 2), fill=LIGHT_BLUE + (115,))

        draw.line((x, y_high, x, y_low), fill=BLUE, width=3)
        draw.line((x - 8, y_high, x + 8, y_high), fill=BLUE, width=3)
        draw.line((x - 8, y_low, x + 8, y_low), fill=BLUE, width=3)
        draw.ellipse((x - 6, y_mean - 6, x + 6, y_mean + 6), fill=BLUE)

        label = row["elo_label"]
        label_image = Image.new("RGBA", (95, 22), (255, 255, 255, 0))
        label_draw = ImageDraw.Draw(label_image)
        label_draw.text((0, 0), label, font=FONT_11, fill=MID)
        rotated = label_image.rotate(330, expand=True)
        image.alpha_composite(rotated, (max(4, int(x - 28)), height - 132))

        profile = profile_by_range.get((row["elo_low"], row["elo_high"]))
        if profile:
            y_profile = scale(profile["avg_acpl"], y_domain, (top + plot_h, top))
            diamond = [(x, y_profile - 11), (x + 11, y_profile), (x, y_profile + 11), (x - 11, y_profile)]
            draw.polygon(diamond, fill=ORANGE, outline=ORANGE_DARK)

    draw.line((left, top, left, top + plot_h), fill=INK, width=2)
    draw.line((left, top + plot_h, left + plot_w, top + plot_h), fill=INK, width=2)
    draw_rotated_text(image, (18, 285), "Average centipawn loss", FONT_13)

    # Clear legend placed in the reserved right margin, not on top of the plot.
    legend_x, legend_y = left + plot_w + 35, top + 20
    draw.rounded_rectangle((legend_x - 15, legend_y - 15, width - 35, legend_y + 118), radius=10, fill=(250, 250, 250, 255), outline=(220, 220, 220, 255))
    draw.text((legend_x, legend_y), "Legend", font=FONT_16, fill=INK)
    draw.ellipse((legend_x, legend_y + 35, legend_x + 13, legend_y + 48), fill=BLUE)
    draw.text((legend_x + 24, legend_y + 32), "Benchmark mean", font=FONT_12, fill=INK)
    draw.line((legend_x + 6, legend_y + 64, legend_x + 6, legend_y + 89), fill=BLUE, width=3)
    draw.text((legend_x + 24, legend_y + 67), "Benchmark ± std", font=FONT_12, fill=INK)
    draw.polygon([(legend_x + 7, legend_y + 104), (legend_x + 17, legend_y + 114), (legend_x + 7, legend_y + 124), (legend_x - 3, legend_y + 114)], fill=ORANGE, outline=ORANGE_DARK)
    draw.text((legend_x + 24, legend_y + 106), "Profile result", font=FONT_12, fill=INK)

    draw_centered_text(draw, (left + plot_w / 2, height - 32), "Elo range", FONT_13, INK)

    path = FIGURE_DIR / f"{DATASET_KIND}_acpl_by_elo.png"
    return save_png(image, path)

acpl_chart_path = acpl_distribution_chart(benchmarks, profiles)
acpl_chart_path


In [ ]:
from __future__ import annotations

def stockfish_maia_scatter(benchmarks: list[dict[str, Any]], profiles: list[dict[str, Any]]) -> Path | None:
    rows = benchmarks + profiles
    if not rows:
        print("No rows available for Stockfish-vs-Maia scatter.")
        return None

    width, height = 1040, 760
    left, right, top, bottom = 90, 300, 95, 90
    plot_w, plot_h = width - left - right, height - top - bottom
    image = Image.new("RGBA", (width, height), WHITE + (255,))
    draw = ImageDraw.Draw(image)

    x_domain = nice_domain([row["avg_match_pct"] for row in rows], floor=0, ceiling=100)
    y_domain = nice_domain([row["avg_maia_pct"] for row in rows], floor=0, ceiling=100)

    high_stockfish = statistics.mean([r["avg_match_pct"] + r.get("std_match_pct", 0.0) for r in benchmarks]) if benchmarks else 55
    low_maia = statistics.mean([r["avg_maia_pct"] - r.get("std_maia_pct", 0.0) for r in benchmarks]) if benchmarks else 35
    high_stockfish = max(x_domain[0], min(x_domain[1], high_stockfish))
    low_maia = max(y_domain[0], min(y_domain[1], low_maia))

    x_ref = scale(high_stockfish, x_domain, (left, left + plot_w))
    y_ref = scale(low_maia, y_domain, (top + plot_h, top))

    draw.text((left, 28), "Stockfish match % vs Maia match %", font=FONT_22, fill=INK)
    draw.text((left, 58), "Review zone: high Stockfish agreement with low Maia plausibility.", font=FONT_13, fill=MID)
    draw.rectangle((x_ref, y_ref, left + plot_w, top + plot_h), fill=REVIEW_FILL + (95,))

    for tick in axis_labels(x_domain):
        x = scale(tick, x_domain, (left, left + plot_w))
        draw.line((x, top, x, top + plot_h), fill=(238, 238, 238), width=1)
        draw_centered_text(draw, (x, top + plot_h + 24), f"{tick:.0f}", FONT_11, MID)
    for tick in axis_labels(y_domain):
        y = scale(tick, y_domain, (top + plot_h, top))
        draw.line((left, y, left + plot_w, y), fill=(238, 238, 238), width=1)
        label = f"{tick:.0f}"
        w, h = text_size(draw, label, FONT_11)
        draw.text((left - w - 10, y - h / 2), label, font=FONT_11, fill=MID)

    # Reference lines marking the review quadrant.
    for offset in range(0, int(plot_h), 14):
        draw.line((x_ref, top + offset, x_ref, min(top + offset + 7, top + plot_h)), fill=ORANGE, width=2)
    for offset in range(0, int(plot_w), 14):
        draw.line((left + offset, y_ref, min(left + offset + 7, left + plot_w), y_ref), fill=ORANGE, width=2)

    for row in benchmarks:
        x = scale(row["avg_match_pct"], x_domain, (left, left + plot_w))
        y = scale(row["avg_maia_pct"], y_domain, (top + plot_h, top))
        draw.ellipse((x - 7, y - 7, x + 7, y + 7), fill=BLUE + (220,))

    for row in profiles:
        x = scale(row["avg_match_pct"], x_domain, (left, left + plot_w))
        y = scale(row["avg_maia_pct"], y_domain, (top + plot_h, top))
        draw.rectangle((x - 8, y - 8, x + 8, y + 8), fill=ORANGE, outline=ORANGE_DARK, width=2)

    draw.line((left, top, left, top + plot_h), fill=INK, width=2)
    draw.line((left, top + plot_h, left + plot_w, top + plot_h), fill=INK, width=2)
    draw_centered_text(draw, (left + plot_w / 2, height - 32), "Stockfish top-move match %", FONT_13, INK)
    draw_rotated_text(image, (18, 350), "Maia top-move match %", FONT_13)

    # Clear legend in the reserved right margin.
    legend_x, legend_y = left + plot_w + 35, top + 20
    draw.rounded_rectangle((legend_x - 15, legend_y - 15, width - 35, legend_y + 192), radius=10, fill=(250, 250, 250, 255), outline=(220, 220, 220, 255))
    draw.text((legend_x, legend_y), "Legend", font=FONT_16, fill=INK)
    draw.ellipse((legend_x, legend_y + 36, legend_x + 14, legend_y + 50), fill=BLUE)
    draw.text((legend_x + 25, legend_y + 34), "Benchmark Elo range", font=FONT_12, fill=INK)
    draw.rectangle((legend_x, legend_y + 71, legend_x + 14, legend_y + 85), fill=ORANGE, outline=ORANGE_DARK, width=2)
    draw.text((legend_x + 25, legend_y + 69), "Profile result", font=FONT_12, fill=INK)
    draw.rectangle((legend_x, legend_y + 108, legend_x + 42, legend_y + 132), fill=REVIEW_FILL + (160,), outline=(230, 160, 95, 255))
    draw.text((legend_x + 55, legend_y + 107), "Review zone", font=FONT_12, fill=INK)
    draw.text((legend_x + 55, legend_y + 124), "high Stockfish / low Maia", font=FONT_10, fill=MID)
    draw.text((legend_x, legend_y + 150), "Point labels removed", font=FONT_10, fill=MID)
    draw.text((legend_x, legend_y + 165), "to avoid overlap.", font=FONT_10, fill=MID)

    path = FIGURE_DIR / f"{DATASET_KIND}_stockfish_vs_maia.png"
    return save_png(image, path)

scatter_chart_path = stockfish_maia_scatter(benchmarks, profiles)
scatter_chart_path


In [ ]:
from __future__ import annotations

def select_profile_with_detail(profiles: list[dict[str, Any]]) -> dict[str, Any] | None:
    for profile in profiles:
        if profile.get("per_game") or profile.get("move_review"):
            return profile
    return profiles[0] if profiles else None

selected_profile = select_profile_with_detail(profiles)
if selected_profile:
    print(f"Selected profile for detail charts: {selected_profile['elo_label']} from {selected_profile['path']}")
else:
    print("No profile result files found.")


def timeline_chart(profile: dict[str, Any] | None) -> Path | None:
    if not profile or not profile.get("per_game"):
        print("Per-game timeline skipped: result JSON does not contain a non-empty 'per_game' list. Rerun profile/general_profile after the latest code changes to populate it.")
        return None

    per_game = profile["per_game"]
    metrics = [
        ("acpl", "ACPL", BLUE),
        ("accuracy", "Accuracy", GREEN),
        ("top_move_pct", "Stockfish top-move %", ORANGE),
        ("maia_matching_pct", "Maia match %", PURPLE),
    ]
    width, height = 1240, 780
    left, right, top, bottom = 100, 70, 90, 70
    image = Image.new("RGBA", (width, height), WHITE + (255,))
    draw = ImageDraw.Draw(image)
    plot_w = width - left - right
    panel_h = (height - top - bottom) / len(metrics)
    x_domain = (1, max(1, len(per_game)))

    draw.text((left, 28), f"Per-game timeline for Elo {profile['elo_label']}", font=FONT_22, fill=INK)
    draw.text((left, 58), "Each panel plots one metric across analyzed player-side samples.", font=FONT_13, fill=MID)

    for panel_index, (field, label, color) in enumerate(metrics):
        y_top = top + panel_index * panel_h
        y_bottom = y_top + panel_h - 22
        values = [game.get(field, 0.0) for game in per_game]
        y_domain = nice_domain(values, floor=0, ceiling=100 if field != "acpl" else None)
        for tick in axis_labels(y_domain, 3):
            y = scale(tick, y_domain, (y_bottom, y_top + 12))
            draw.line((left, y, width - right, y), fill=(238, 238, 238), width=1)
            label_tick = f"{tick:.0f}"
            w, h = text_size(draw, label_tick, FONT_10)
            draw.text((left - w - 10, y - h / 2), label_tick, font=FONT_10, fill=MID)

        points = []
        for i, value in enumerate(values, 1):
            x = scale(i, x_domain, (left, width - right))
            y = scale(value, y_domain, (y_bottom, y_top + 12))
            points.append((x, y))
        if len(points) > 1:
            draw.line(points, fill=color, width=3)
        for x, y in points:
            draw.ellipse((x - 3, y - 3, x + 3, y + 3), fill=color)
        draw_rotated_text(image, (20, int(y_top + panel_h / 2 - 70)), label, FONT_12)

    draw_centered_text(draw, (left + plot_w / 2, height - 28), "Analyzed player-side sample order", FONT_13, INK)
    path = FIGURE_DIR / f"{DATASET_KIND}_{profile['elo_low']}_{profile['elo_high']}_timeline.png"
    return save_png(image, path)


def phase_breakdown_chart(profile: dict[str, Any] | None) -> Path | None:
    if not profile or not profile.get("per_game"):
        print("Phase breakdown skipped: result JSON does not contain per-game phase summaries yet.")
        return None

    phases = ["opening", "middlegame", "endgame"]
    phase_rows = []
    for phase in phases:
        acpls = []
        for game in profile["per_game"]:
            if phase in game.get("acpl_by_phase", {}):
                acpls.append(game["acpl_by_phase"][phase])
        phase_rows.append({
            "phase": phase,
            "acpl": statistics.mean(acpls) if acpls else 0.0,
            "sample_count": len(acpls),
        })

    width, height = 900, 540
    left, right, top, bottom = 95, 45, 90, 80
    plot_w, plot_h = width - left - right, height - top - bottom
    image = Image.new("RGBA", (width, height), WHITE + (255,))
    draw = ImageDraw.Draw(image)
    y_domain = nice_domain([row["acpl"] for row in phase_rows], floor=0)

    draw.text((left, 28), f"Phase ACPL breakdown for Elo {profile['elo_label']}", font=FONT_22, fill=INK)
    draw.text((left, 58), "Higher middlegame precision than surrounding phases can be worth deeper review.", font=FONT_13, fill=MID)

    for tick in axis_labels(y_domain):
        y = scale(tick, y_domain, (top + plot_h, top))
        draw.line((left, y, left + plot_w, y), fill=(238, 238, 238), width=1)
        label_tick = f"{tick:.0f}"
        w, h = text_size(draw, label_tick, FONT_11)
        draw.text((left - w - 10, y - h / 2), label_tick, font=FONT_11, fill=MID)

    gap = plot_w / len(phase_rows)
    bar_w = 80
    for index, row in enumerate(phase_rows):
        x = left + index * gap + gap / 2 - bar_w / 2
        y = scale(row["acpl"], y_domain, (top + plot_h, top))
        draw.rectangle((x, y, x + bar_w, top + plot_h), fill=BLUE + (225,))
        draw_centered_text(draw, (x + bar_w / 2, y - 12), f"{row['acpl']:.1f}", FONT_11, INK)
        draw_centered_text(draw, (x + bar_w / 2, height - 45), row["phase"], FONT_12, INK)
        draw_centered_text(draw, (x + bar_w / 2, height - 25), f"n={row['sample_count']}", FONT_10, MID)

    draw.line((left, top, left, top + plot_h), fill=INK, width=2)
    draw.line((left, top + plot_h, left + plot_w, top + plot_h), fill=INK, width=2)
    draw_rotated_text(image, (18, 250), "Average ACPL", FONT_13)
    path = FIGURE_DIR / f"{DATASET_KIND}_{profile['elo_low']}_{profile['elo_high']}_phase_acpl.png"
    return save_png(image, path)


def move_review_table(profile: dict[str, Any] | None, limit: int = 80) -> Path | None:
    if not profile or not profile.get("move_review"):
        print("Move-level table skipped: result JSON does not contain move_review rows yet. Rerun profile/general_profile after the latest code changes and with Maia enabled for Maia predicted moves.")
        return None

    rows = sorted(profile["move_review"], key=lambda row: (not row.get("review_flag", False), -float(row.get("cp_loss", 0))))[:limit]
    headers = ["game", "player", "elo", "color", "ply", "phase", "played", "stockfish best", "maia predicted", "cp loss", "stockfish match", "maia match", "flag"]
    html_rows = ["<table>", "<thead><tr>" + "".join(f"<th>{esc(h)}</th>" for h in headers) + "</tr></thead>", "<tbody>"]
    for row in rows:
        html_rows.append("<tr>" + "".join([
            f"<td>{esc(row.get('game_index', ''))}</td>",
            f"<td>{esc(row.get('player', ''))}</td>",
            f"<td>{esc(row.get('elo', ''))}</td>",
            f"<td>{esc(row.get('color', ''))}</td>",
            f"<td>{esc(row.get('ply', ''))}</td>",
            f"<td>{esc(row.get('phase', ''))}</td>",
            f"<td>{esc(row.get('played_move', ''))}</td>",
            f"<td>{esc(row.get('stockfish_best_move', ''))}</td>",
            f"<td>{esc(row.get('maia_predicted_move', ''))}</td>",
            f"<td>{float(row.get('cp_loss', 0)):.1f}</td>",
            f"<td>{esc(row.get('stockfish_match', ''))}</td>",
            f"<td>{esc(row.get('maia_match', ''))}</td>",
            f"<td>{'review' if row.get('review_flag') else ''}</td>",
        ]) + "</tr>")
    html_rows.append("</tbody></table>")
    table_html = """
<style>
table { border-collapse: collapse; font-family: Arial, sans-serif; font-size: 12px; }
th, td { border: 1px solid #ddd; padding: 5px 7px; text-align: left; }
th { background: #f4f4f4; }
tr:nth-child(even) { background: #fafafa; }
</style>
""" + "\n".join(html_rows)
    path = REPORT_DIR / f"{DATASET_KIND}_{profile['elo_low']}_{profile['elo_high']}_move_review.html"
    write_text(path, table_html)
    show_html(table_html)
    return path

timeline_chart_path = timeline_chart(selected_profile)
phase_chart_path = phase_breakdown_chart(selected_profile)
move_table_path = move_review_table(selected_profile)
(timeline_chart_path, phase_chart_path, move_table_path)


In [ ]:
from __future__ import annotations

def relative_report_link(path: Path | None) -> str | None:
    if path is None:
        return None
    return path.relative_to(REPORT_DIR).as_posix()


def build_markdown_report() -> Path:
    lines = [
        "# Chess cheat-detection visualization report",
        "",
        f"Dataset kind: `{DATASET_KIND}`",
        "",
        "This report is generated by `notebooks/reviewer_visualizations.ipynb`.",
        "It is meant for human review of benchmark/profile outputs, not for automatic accusations.",
        "",
        "## Available charts",
        "",
    ]

    if acpl_chart_path:
        lines.extend([
            "### ACPL by Elo range",
            "",
            f"![ACPL by Elo range]({relative_report_link(acpl_chart_path)})",
            "",
        ])
    else:
        lines.append("- ACPL chart was not generated because no benchmark data was found.")

    if scatter_chart_path:
        lines.extend([
            "### Stockfish match % vs Maia match %",
            "",
            f"![Stockfish vs Maia scatter]({relative_report_link(scatter_chart_path)})",
            "",
        ])
    else:
        lines.append("- Stockfish-vs-Maia scatter was not generated because no benchmark/profile data was found.")

    if timeline_chart_path:
        lines.extend([
            "### Per-game timeline",
            "",
            f"![Per-game timeline]({relative_report_link(timeline_chart_path)})",
            "",
        ])
    else:
        lines.extend([
            "### Per-game timeline",
            "",
            "Not generated yet. The selected profile JSON needs a `per_game` list. Rerun `profile` or `general_profile` after the latest code changes to save this detail.",
            "",
        ])

    if phase_chart_path:
        lines.extend([
            "### Phase breakdown",
            "",
            f"![Phase breakdown]({relative_report_link(phase_chart_path)})",
            "",
        ])
    else:
        lines.extend([
            "### Phase breakdown",
            "",
            "Not generated yet. The selected profile JSON needs `per_game[*].acpl_by_phase` and `per_game[*].top_move_pct_by_phase`.",
            "",
        ])

    if move_table_path:
        lines.extend([
            "### Move-level review table",
            "",
            f"Open the move review table: [{move_table_path.name}]({relative_report_link(move_table_path)})",
            "",
        ])
    else:
        lines.extend([
            "### Move-level review table",
            "",
            "Not generated yet. The selected profile JSON needs a `move_review` list. Rerun the profiling command after the latest code changes. Use Maia if you want the Maia predicted move column filled.",
            "",
        ])

    lines.extend([
        "## Reading notes",
        "",
        "- Low ACPL is strong play, but it is not suspicious by itself at high Elo.",
        "- High Stockfish match with low Maia match is more interesting than either metric alone.",
        "- Phase and move-level views are for reviewer triage: they show where to look, not what conclusion to draw.",
        "- Existing older JSON files may only support summary-level charts. Rerunning the CLI with the updated code will create richer profile JSONs.",
        "",
    ])

    path = REPORT_DIR / "README.md"
    write_text(path, "\n".join(lines))
    return path

report_path = build_markdown_report()
print(f"Wrote report: {report_path}")
report_path


## Takeaways

Use this notebook as the visual review layer:

- Benchmark files define the rating-relative baseline.
- Profile files show the sample being reviewed.
- Rich profile outputs unlock the timeline, phase, and move-level sections.

The first two charts work with the existing summary files. The last three become useful after rerunning profile commands so the result JSON includes `per_game` and `move_review` detail.
